# EPLAN Electric P8 — Fine-Tune LLM with Unsloth

Fine-tunes a 7B model on 4,288 Q&A pairs from EPLAN documentation.

**Runtime:** Kaggle GPU T4 x2 (free tier)
**Time:** ~1-2 hours

## Steps
1. Install Unsloth
2. Load dataset from HuggingFace
3. Fine-tune with LoRA (4-bit quantization)
4. Export to GGUF for Ollama
5. Push to HuggingFace Hub

## 1. Install Dependencies

In [ ]:
%%capture
!pip install unsloth
!pip uninstall unsloth -y && pip install --upgrade --no-cache-dir --no-deps git+https://github.com/unslothai/unsloth.git

## 2. Load Model

In [ ]:
from unsloth import FastLanguageModel
import torch

# === CHOOSE YOUR BASE MODEL ===
# Option A: Qwen 2.5 7B (best quality, fits T4 16GB with 4-bit)
MODEL_NAME = "unsloth/Qwen2.5-7B-Instruct-bnb-4bit"

# Option B: Llama 3.1 8B (alternative)
# MODEL_NAME = "unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit"

# Option C: Phi-3.5 Mini (faster, lighter, lower quality)
# MODEL_NAME = "unsloth/Phi-3.5-mini-instruct-bnb-4bit"

MAX_SEQ_LENGTH = 4096

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,  # auto-detect
    load_in_4bit=True,
)

## 3. Configure LoRA

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,                     # LoRA rank (16 is good balance)
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0,           # Unsloth optimized: 0 is fastest
    bias="none",
    use_gradient_checkpointing="unsloth",  # 60% less VRAM
    random_state=42,
)

# Show trainable params
model.print_trainable_parameters()

## 4. Load & Format Dataset

In [ ]:
from datasets import load_dataset

# Load from HuggingFace Hub
dataset = load_dataset("covaga/eplan-qa-dataset", data_files="eplan_qa_FINAL.jsonl", split="train")

print(f"Dataset: {len(dataset)} examples")
print(dataset[0])

In [ ]:
# Format into chat template
SYSTEM_MSG = """You are an expert EPLAN Electric P8 assistant specialized in industrial electrical engineering. 
You help engineers with API usage, troubleshooting, best practices, and procedural guidance. 
Provide accurate, detailed answers based on EPLAN documentation."""

def format_chat(example):
    """Convert instruction/output to chat format."""
    messages = [
        {"role": "system", "content": SYSTEM_MSG},
        {"role": "user", "content": example["instruction"]},
        {"role": "assistant", "content": example["output"]},
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
    return {"text": text}

dataset = dataset.map(format_chat)
print("Sample formatted:")
print(dataset[0]["text"][:500])

## 5. Train

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LENGTH,
    dataset_num_proc=2,
    packing=True,  # Pack short examples together for efficiency
    args=TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,  # Effective batch = 8
        warmup_steps=10,
        num_train_epochs=3,
        learning_rate=2e-4,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=10,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=42,
        output_dir="outputs",
        save_strategy="epoch",
    ),
)

print("Starting training...")
stats = trainer.train()
print(f"\nTraining complete!")
print(f"  Loss: {stats.training_loss:.4f}")
print(f"  Runtime: {stats.metrics['train_runtime']:.0f}s")

## 6. Test the Model

In [ ]:
# Quick test
FastLanguageModel.for_inference(model)

test_questions = [
    "How do I export a project to PDF using the EPLAN API?",
    "What is the difference between a Function and a FunctionBase?",
    "My script throws NullReferenceException when iterating pages. What could cause this?",
]

for q in test_questions:
    messages = [
        {"role": "system", "content": SYSTEM_MSG},
        {"role": "user", "content": q},
    ]
    inputs = tokenizer.apply_chat_template(messages, tokenize=True, add_generation_prompt=True, return_tensors="pt").to("cuda")
    outputs = model.generate(input_ids=inputs, max_new_tokens=512, temperature=0.7)
    response = tokenizer.decode(outputs[0][inputs.shape[-1]:], skip_special_tokens=True)
    print(f"Q: {q}")
    print(f"A: {response}")
    print("-" * 80)

## 7. Export to GGUF (for Ollama)

In [ ]:
# Save GGUF quantized versions
# Q4_K_M = best balance of size vs quality (~4.5GB)
# Q8_0 = higher quality, bigger (~8GB)

model.save_pretrained_gguf(
    "eplan-assistant-gguf",
    tokenizer,
    quantization_method="q4_k_m",  # ~4.5GB, good for your 4GB target
)
print("GGUF Q4_K_M saved!")

## 8. Push to HuggingFace Hub

In [ ]:
# === Login to HuggingFace ===
from huggingface_hub import login
# Get your token from https://huggingface.co/settings/tokens
# In Kaggle: Settings > Secrets > add HF_TOKEN
from kaggle_secrets import UserSecretsClient
login(token=UserSecretsClient().get_secret("HF_TOKEN"))

In [ ]:
# Push LoRA adapter (small, ~50MB)
model.push_to_hub("covaga/eplan-assistant-lora", tokenizer=tokenizer)
print("LoRA adapter pushed!")

# Push GGUF (for Ollama, ~4.5GB)
model.push_to_hub_gguf(
    "covaga/eplan-assistant-gguf",
    tokenizer,
    quantization_method="q4_k_m",
)
print("GGUF pushed!")

## 9. Use in Ollama (local)

After downloading the GGUF from HuggingFace:

```bash
# Create Modelfile
cat > Modelfile << 'EOF'
FROM ./eplan-assistant-q4_k_m.gguf

SYSTEM """You are an expert EPLAN Electric P8 assistant specialized in industrial electrical engineering.
You help engineers with API usage, troubleshooting, best practices, and procedural guidance.
Provide accurate, detailed answers based on EPLAN documentation."""

PARAMETER temperature 0.7
PARAMETER top_p 0.9
EOF

# Create and run
ollama create eplan-assistant -f Modelfile
ollama run eplan-assistant
```